In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# -------------------------
# Tables
# -------------------------
bronze_tbl = "capstone.bronze.geography"
silver_tbl = "capstone.silver.geography"

df = spark.table(bronze_tbl)

# -------------------------
# 1. Select needed columns
# -------------------------
df = df.select(
    "geo_id",
    "city",
    "county",
    "state",
    "region",
    "ingestion_timestamp"
)

# -------------------------
# 2. Basic cleaning
# -------------------------
df = df.withColumn("city", F.initcap(F.trim("city"))) \
       .withColumn("county", F.initcap(F.trim("county"))) \
       .withColumn("state", F.upper(F.trim("state"))) \
       .withColumn("region", F.initcap(F.trim("region")))

# -------------------------
# 3. Standardize region
# -------------------------
df = df.withColumn(
    "region",
    F.when(F.col("region").isin("Ne","North East","Northeast"), "Northeast")
     .when(F.col("region").isin("Mw","Mid West","Midwest"), "Midwest")
     .when(F.col("region").isin("South","Southeast"), "South")
     .when(F.col("region").isin("West","W"), "West")
     .otherwise(F.col("region"))
)

# -------------------------
# 4. Remove null geo_id
# -------------------------
df = df.filter(F.col("geo_id").isNotNull())

# -------------------------
# 5. Deduplication
# -------------------------
window = Window.partitionBy("geo_id").orderBy(F.col("ingestion_timestamp").desc())

df = df.withColumn("row_num", F.row_number().over(window)) \
       .filter(F.col("row_num") == 1) \
       .drop("row_num")

# -------------------------
# 6. Write to Silver
# -------------------------
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(silver_tbl)

In [0]:
%sql
-- ===============================
-- 1. Total Rows
-- ===============================
SELECT 
'Total Rows' AS check_type,
COUNT(*) AS result
FROM capstone.silver.geography

UNION ALL

-- ===============================
-- 2. Missing Geo ID
-- ===============================
SELECT
'Missing Geo ID',
COUNT(*)
FROM capstone.silver.geography
WHERE geo_id IS NULL

UNION ALL

-- ===============================
-- 3. Missing City
-- ===============================
SELECT
'Missing City',
COUNT(*)
FROM capstone.silver.geography
WHERE city IS NULL OR TRIM(city) = ''

UNION ALL

-- ===============================
-- 4. Missing County
-- ===============================
SELECT
'Missing County',
COUNT(*)
FROM capstone.silver.geography
WHERE county IS NULL OR TRIM(county) = ''

UNION ALL

-- ===============================
-- 5. Missing State
-- ===============================
SELECT
'Missing State',
COUNT(*)
FROM capstone.silver.geography
WHERE state IS NULL OR TRIM(state) = ''

UNION ALL

-- ===============================
-- 6. Missing Region
-- ===============================
SELECT
'Missing Region',
COUNT(*)
FROM capstone.silver.geography
WHERE region IS NULL OR TRIM(region) = ''

UNION ALL

-- ===============================
-- 7. Invalid Region Values
-- ===============================
SELECT
'Invalid Region Values',
COUNT(*)
FROM capstone.silver.geography
WHERE region NOT IN ('Northeast','Midwest','South','West')

UNION ALL

-- ===============================
-- 8. Duplicate Geo IDs
-- ===============================
SELECT
'Duplicate Geo IDs',
COUNT(*)
FROM (
    SELECT geo_id
    FROM capstone.silver.geography
    GROUP BY geo_id
    HAVING COUNT(*) > 1
)

UNION ALL

-- ===============================
-- 9. Duplicate City/County/State
-- ===============================
SELECT
'Duplicate Geography Combination',
COUNT(*)
FROM (
    SELECT city, county, state
    FROM capstone.silver.geography
    GROUP BY city, county, state
    HAVING COUNT(*) > 1
)

UNION ALL

-- ===============================
-- 10. Missing Ingestion Timestamp
-- ===============================
SELECT
'Missing Ingestion Timestamp',
COUNT(*)
FROM capstone.silver.geography
WHERE ingestion_timestamp IS NULL;